# 🛒 Flipkart AI Product Recommender
> **RAG-powered e-commerce assistant — LangGraph ReAct · FAISS · HuggingFace Embeddings · Groq LLM**

---

## 🔑 APIs Required — Only 2

| Service | What for | Get it |
|---|---|---|
| **Groq** | LLM inference (`llama-3.1-8b-instant`) | https://console.groq.com — free |
| **HuggingFace** | Embedding model download (`BAAI/bge-base-en-v1.5`) | https://huggingface.co/settings/tokens — free |

> ✅ **No AstraDB, no Pinecone, no external vector DB** — uses local **FAISS** index in Colab RAM.

---

## 📁 Project Structure
```
Flipkart/
├── flipkart/
│   ├── __init__.py
│   ├── config.py           # Groq + embedding config
│   ├── data_converter.py   # Stage 2: CSV → LangChain Documents
│   ├── data_ingestion.py   # Stage 3: FAISS vector store build
│   └── rag_agent.py        # Stage 4: LangGraph ReAct agent
├── utils/
│   ├── logger.py
│   └── custom_exception.py
├── data/
│   └── flipkart_product_review.csv
├── streamlit_app.py        # Streamlit UI (reference)
└── outputs/
```

## 🚀 Pipeline
```
CSV Data → DataConverter → DataIngestor (FAISS) → RAGAgent (LangGraph) → Answers
```


---
## 🔧 Stage 0 — Install Dependencies
Using the same pinned versions confirmed working with **Groq + LangGraph on Colab**.


In [1]:
import base64, os, subprocess, sys, uuid  # uuid in global scope — fixes langchain-core NameError

_req_b64 = "bGFuZ2NoYWluPT0wLjMuMjcKbGFuZ2NoYWluLWNvbW11bml0eT09MC4zLjI3CmxhbmdjaGFpbi1ncm9xCmxhbmdjaGFpbi1odWdnaW5nZmFjZQpsYW5nZ3JhcGg9PTAuNi41CmZhaXNzLWNwdQpweWRhbnRpYwpweXBkZgpweXRob24tZG90ZW52PT0xLjIuMQpwYW5kYXM9PTIuMy4zCnNlbnRlbmNlLXRyYW5zZm9ybWVycwpzdHJlYW1saXQK"
with open("requirements.txt", "w") as f:
    f.write(base64.b64decode(_req_b64).decode())

print("📦 Installing dependencies (2-4 min on first run)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
else:
    print("✅ All dependencies installed successfully")


📦 Installing dependencies (2-4 min on first run)...
✅ All dependencies installed successfully


---
## 📁 Stage 0b — Write All Source Files to Disk
All `.py` source files are base64-embedded and decoded at runtime.


In [2]:
import os, base64, sys

for d in ["flipkart", "utils", "data", "logs", "outputs"]:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, ".")

_FILES_B64 = {
    "flipkart/__init__.py": "",
    "flipkart/config.py": "aW1wb3J0IG9zCmZyb20gZG90ZW52IGltcG9ydCBsb2FkX2RvdGVudgpmcm9tIGxhbmdjaGFpbl9ncm9xIGltcG9ydCBDaGF0R3JvcQoKbG9hZF9kb3RlbnYoKQoKCmNsYXNzIENvbmZpZzoKICAgIEdST1FfQVBJX0tFWSAgICA9IG9zLmdldGVudigiR1JPUV9BUElfS0VZIiwgIiIpCiAgICBIRl9UT0tFTiAgICAgICAgPSBvcy5nZXRlbnYoIkhGX1RPS0VOIiwgIiIpCiAgICBMTE1fTU9ERUwgICAgICAgPSAibGxhbWEtMy4xLThiLWluc3RhbnQiCiAgICBFTUJFRERJTkdfTU9ERUwgPSAiYWxsLU1pbmlMTS1MNi12MiIKICAgIENIVU5LX1NJWkUgICAgICA9IDUwMAogICAgQ0hVTktfT1ZFUkxBUCAgID0gNTAKCiAgICBAY2xhc3NtZXRob2QKICAgIGRlZiBnZXRfbGxtKGNscyk6CiAgICAgICAgcmV0dXJuIENoYXRHcm9xKAogICAgICAgICAgICBtb2RlbD1jbHMuTExNX01PREVMLAogICAgICAgICAgICBhcGlfa2V5PWNscy5HUk9RX0FQSV9LRVksCiAgICAgICAgICAgIHRlbXBlcmF0dXJlPTAsCiAgICAgICAgKQo=",
    "flipkart/data_converter.py": "aW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIGxhbmdjaGFpbl9jb3JlLmRvY3VtZW50cyBpbXBvcnQgRG9jdW1lbnQKCgpjbGFzcyBEYXRhQ29udmVydGVyOgogICAgIiIiQ29udmVydHMgdGhlIEZsaXBrYXJ0IENTViBpbnRvIExhbmdDaGFpbiBEb2N1bWVudHMuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGZpbGVfcGF0aDogc3RyKToKICAgICAgICBzZWxmLmZpbGVfcGF0aCA9IGZpbGVfcGF0aAoKICAgIGRlZiBjb252ZXJ0KHNlbGYpOgogICAgICAgIGRmID0gcGQucmVhZF9jc3Yoc2VsZi5maWxlX3BhdGgpW1sicHJvZHVjdF90aXRsZSIsICJyZXZpZXciXV0uZHJvcG5hKCkKCiAgICAgICAgZG9jcyA9IFsKICAgICAgICAgICAgRG9jdW1lbnQoCiAgICAgICAgICAgICAgICBwYWdlX2NvbnRlbnQ9cm93WyJyZXZpZXciXSwKICAgICAgICAgICAgICAgIG1ldGFkYXRhPXsicHJvZHVjdF9uYW1lIjogcm93WyJwcm9kdWN0X3RpdGxlIl19CiAgICAgICAgICAgICkKICAgICAgICAgICAgZm9yIF8sIHJvdyBpbiBkZi5pdGVycm93cygpCiAgICAgICAgXQoKICAgICAgICByZXR1cm4gZG9jcwo=",
    "flipkart/data_ingestion.py": "ZnJvbSBsYW5nY2hhaW5fY29tbXVuaXR5LnZlY3RvcnN0b3JlcyBpbXBvcnQgRkFJU1MKZnJvbSBsYW5nY2hhaW5faHVnZ2luZ2ZhY2UgaW1wb3J0IEh1Z2dpbmdGYWNlRW1iZWRkaW5ncwpmcm9tIGxhbmdjaGFpbl90ZXh0X3NwbGl0dGVycyBpbXBvcnQgUmVjdXJzaXZlQ2hhcmFjdGVyVGV4dFNwbGl0dGVyCmZyb20gZmxpcGthcnQuZGF0YV9jb252ZXJ0ZXIgaW1wb3J0IERhdGFDb252ZXJ0ZXIKZnJvbSBmbGlwa2FydC5jb25maWcgaW1wb3J0IENvbmZpZwoKCmNsYXNzIERhdGFJbmdlc3RvcjoKICAgICIiIgogICAgRW1iZWRzIEZsaXBrYXJ0IHByb2R1Y3QgcmV2aWV3cyBpbnRvIGEgbG9jYWwgRkFJU1MgdmVjdG9yIHN0b3JlLgogICAgTm8gZXh0ZXJuYWwgdmVjdG9yIERCIEFQSSByZXF1aXJlZCDigJQgcnVucyBmdWxseSBvbiBDb2xhYi4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmKToKICAgICAgICBwcmludCgiICDwn5SEIExvYWRpbmcgZW1iZWRkaW5nIG1vZGVsOiBCQUFJL2JnZS1iYXNlLWVuLXYxLjUgLi4uIikKICAgICAgICBzZWxmLmVtYmVkZGluZyA9IEh1Z2dpbmdGYWNlRW1iZWRkaW5ncygKICAgICAgICAgICAgbW9kZWxfbmFtZT0iQkFBSS9iZ2UtYmFzZS1lbi12MS41IgogICAgICAgICkKICAgICAgICBzZWxmLnNwbGl0dGVyID0gUmVjdXJzaXZlQ2hhcmFjdGVyVGV4dFNwbGl0dGVyKAogICAgICAgICAgICBjaHVua19zaXplPUNvbmZpZy5DSFVOS19TSVpFLAogICAgICAgICAgICBjaHVua19vdmVybGFwPUNvbmZpZy5DSFVOS19PVkVSTEFQLAogICAgICAgICkKICAgICAgICBzZWxmLnZlY3RvcnN0b3JlID0gTm9uZQogICAgICAgIHNlbGYucmV0cmlldmVyICAgPSBOb25lCiAgICAgICAgcHJpbnQoIiAg4pyFIEVtYmVkZGluZyBtb2RlbCBsb2FkZWQiKQoKICAgIGRlZiBpbmdlc3Qoc2VsZiwgY3N2X3BhdGg6IHN0ciA9ICJkYXRhL2ZsaXBrYXJ0X3Byb2R1Y3RfcmV2aWV3LmNzdiIpOgogICAgICAgICIiIgogICAgICAgIExvYWQgQ1NWIOKGkiBjb252ZXJ0IHRvIERvY3VtZW50cyDihpIgZW1iZWQg4oaSIGJ1aWxkIEZBSVNTIGluZGV4LgogICAgICAgIFJ1biBvbmNlOyB0aGUgaW5kZXggbGl2ZXMgaW4gbWVtb3J5IGZvciB0aGUgc2Vzc2lvbi4KICAgICAgICAiIiIKICAgICAgICBkb2NzICAgPSBEYXRhQ29udmVydGVyKGNzdl9wYXRoKS5jb252ZXJ0KCkKICAgICAgICBjaHVua3MgPSBzZWxmLnNwbGl0dGVyLnNwbGl0X2RvY3VtZW50cyhkb2NzKQogICAgICAgIHByaW50KGYiICDinILvuI8gIHtsZW4oZG9jcyl9IHJvd3Mg4oaSIHtsZW4oY2h1bmtzKX0gY2h1bmtzIikKCiAgICAgICAgcHJpbnQoZiIgIPCflI0gQnVpbGRpbmcgRkFJU1MgaW5kZXggb3ZlciB7bGVuKGNodW5rcyl9IGNodW5rcyAuLi4iKQogICAgICAgIHNlbGYudmVjdG9yc3RvcmUgPSBGQUlTUy5mcm9tX2RvY3VtZW50cyhjaHVua3MsIHNlbGYuZW1iZWRkaW5nKQogICAgICAgIHNlbGYucmV0cmlldmVyICAgPSBzZWxmLnZlY3RvcnN0b3JlLmFzX3JldHJpZXZlcigKICAgICAgICAgICAgc2VhcmNoX2t3YXJncz17ImsiOiA0fQogICAgICAgICkKICAgICAgICBwcmludCgiICDinIUgRkFJU1MgaW5kZXggcmVhZHkiKQogICAgICAgIHJldHVybiBzZWxmLnJldHJpZXZlcgo=",
    "flipkart/rag_agent.py": "ZnJvbSB0eXBpbmcgaW1wb3J0IExpc3QsIE9wdGlvbmFsCmZyb20gbGFuZ2NoYWluX2NvcmUuZG9jdW1lbnRzIGltcG9ydCBEb2N1bWVudApmcm9tIGxhbmdjaGFpbl9jb3JlLm1lc3NhZ2VzIGltcG9ydCAoCiAgICBIdW1hbk1lc3NhZ2UsIFN5c3RlbU1lc3NhZ2UsIFRvb2xNZXNzYWdlLCBBSU1lc3NhZ2UKKQpmcm9tIGxhbmdjaGFpbl9jb3JlLnRvb2xzIGltcG9ydCB0b29sCmZyb20gbGFuZ2dyYXBoLmdyYXBoIGltcG9ydCBTdGF0ZUdyYXBoLCBFTkQKZnJvbSBweWRhbnRpYyBpbXBvcnQgQmFzZU1vZGVsLCBDb25maWdEaWN0CmZyb20gZmxpcGthcnQuY29uZmlnIGltcG9ydCBDb25maWcKCgojIOKUgOKUgCBUeXBlZCBzdGF0ZSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKY2xhc3MgUkFHU3RhdGUoQmFzZU1vZGVsKToKICAgIHF1ZXN0aW9uICAgICAgOiBzdHIKICAgIHJldHJpZXZlZF9kb2NzOiBMaXN0W0RvY3VtZW50XSA9IFtdCiAgICBhbnN3ZXIgICAgICAgIDogc3RyID0gIiIKICAgIG1vZGVsX2NvbmZpZyA9IENvbmZpZ0RpY3QoYXJiaXRyYXJ5X3R5cGVzX2FsbG93ZWQ9VHJ1ZSkKCgojIOKUgOKUgCBUb29sIGZhY3Rvcnkg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACmRlZiBfbWFrZV9yZXRyaWV2ZXJfdG9vbChyZXRyaWV2ZXIpOgoKICAgIEB0b29sCiAgICBkZWYgc2VhcmNoX2ZsaXBrYXJ0X3Jldmlld3MocXVlcnk6IHN0cikgLT4gc3RyOgogICAgICAgICIiIgogICAgICAgIFNlYXJjaCBGbGlwa2FydCBwcm9kdWN0IHJldmlld3MgZm9yIGluZm9ybWF0aW9uIHJlbGV2YW50IHRvIHRoZSBxdWVyeS4KICAgICAgICBBbHdheXMgY2FsbCB0aGlzIHRvb2wgZmlyc3QgYmVmb3JlIGFuc3dlcmluZyBhbnkgcHJvZHVjdCBxdWVzdGlvbi4KICAgICAgICAiIiIKICAgICAgICBkb2NzOiBMaXN0W0RvY3VtZW50XSA9IHJldHJpZXZlci5pbnZva2UocXVlcnkpCiAgICAgICAgaWYgbm90IGRvY3M6CiAgICAgICAgICAgIHJldHVybiAiTm8gcmVsZXZhbnQgcHJvZHVjdCByZXZpZXdzIGZvdW5kLiIKICAgICAgICBwYXJ0cyA9IFtdCiAgICAgICAgZm9yIGksIGQgaW4gZW51bWVyYXRlKGRvY3MsIDEpOgogICAgICAgICAgICBwcm9kdWN0ID0gZC5tZXRhZGF0YS5nZXQoInByb2R1Y3RfbmFtZSIsICJVbmtub3duIHByb2R1Y3QiKQogICAgICAgICAgICBwYXJ0cy5hcHBlbmQoZiJbe2l9XSBQcm9kdWN0OiB7cHJvZHVjdH1cblJldmlldzoge2QucGFnZV9jb250ZW50fSIpCiAgICAgICAgcmV0dXJuICJcblxuIi5qb2luKHBhcnRzKQoKICAgIHJldHVybiBzZWFyY2hfZmxpcGthcnRfcmV2aWV3cwoKCiMg4pSA4pSAIFJBR05vZGVzIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgApjbGFzcyBSQUdOb2RlczoKICAgICIiIk5vZGUgZnVuY3Rpb25zIGZvciB0aGUgTGFuZ0dyYXBoIFJBRyB3b3JrZmxvdyAoR3JvcS1zYWZlIHRvb2wgbG9vcCkuIiIiCgogICAgU1lTVEVNX1BST01QVCA9ICgKICAgICAgICAiWW91IGFyZSBhIEZsaXBrYXJ0IGUtY29tbWVyY2UgYXNzaXN0YW50LiAiCiAgICAgICAgIkFsd2F5cyB1c2Ugc2VhcmNoX2ZsaXBrYXJ0X3Jldmlld3MgdG8gbG9vayB1cCByZWxldmFudCBwcm9kdWN0IHJldmlld3MgIgogICAgICAgICJiZWZvcmUgYW5zd2VyaW5nLiBTdW1tYXJpc2UgZmluZGluZ3MgY2xlYXJseSBhbmQgaGVscGZ1bGx5LiAiCiAgICAgICAgIklmIHlvdSBjYW5ub3QgZmluZCBhbiBhbnN3ZXIgc2F5OiAiCiAgICAgICAgIidJIGRvbid0IGhhdmUgZW5vdWdoIGluZm9ybWF0aW9uIG9uIHRoYXQuIFBsZWFzZSBjb250YWN0IGN1c3RvbWVyIGNhcmUuJyIKICAgICkKCiAgICBkZWYgX19pbml0X18oc2VsZiwgcmV0cmlldmVyLCBsbG0sIG1heF9pdGVyYXRpb25zOiBpbnQgPSA1KToKICAgICAgICBzZWxmLnJldHJpZXZlciAgICAgICA9IHJldHJpZXZlcgogICAgICAgIHNlbGYubGxtICAgICAgICAgICAgID0gbGxtCiAgICAgICAgc2VsZi5tYXhfaXRlcmF0aW9ucyAgPSBtYXhfaXRlcmF0aW9ucwogICAgICAgIHNlbGYuX3Rvb2wgICAgICAgICAgID0gX21ha2VfcmV0cmlldmVyX3Rvb2wocmV0cmlldmVyKQogICAgICAgIHNlbGYuX3Rvb2xfbWFwICAgICAgID0ge3NlbGYuX3Rvb2wubmFtZTogc2VsZi5fdG9vbH0KICAgICAgICBzZWxmLl9sbG1fd2l0aF90b29scyA9IGxsbS5iaW5kX3Rvb2xzKFtzZWxmLl90b29sXSkKCiAgICBkZWYgcmV0cmlldmVfZG9jcyhzZWxmLCBzdGF0ZTogUkFHU3RhdGUpIC0+IFJBR1N0YXRlOgogICAgICAgICIiIlByZS1mZXRjaCB0b3AtayBkb2NzIHNvIHN0YXRlLnJldHJpZXZlZF9kb2NzIGlzIGFsd2F5cyBwb3B1bGF0ZWQuIiIiCiAgICAgICAgZG9jcyA9IHNlbGYucmV0cmlldmVyLmludm9rZShzdGF0ZS5xdWVzdGlvbikKICAgICAgICByZXR1cm4gUkFHU3RhdGUocXVlc3Rpb249c3RhdGUucXVlc3Rpb24sIHJldHJpZXZlZF9kb2NzPWRvY3MpCgogICAgZGVmIGdlbmVyYXRlX2Fuc3dlcihzZWxmLCBzdGF0ZTogUkFHU3RhdGUpIC0+IFJBR1N0YXRlOgogICAgICAgICIiIgogICAgICAgIEdyb3EtY29tcGF0aWJsZSBhZ2VudGljIHRvb2wgbG9vcC4KICAgICAgICBVc2VzIGJpbmRfdG9vbHMgKyBtYW51YWwgVG9vbE1lc3NhZ2UgbG9vcCAoYXZvaWRzIEdyb3EgQmFkUmVxdWVzdEVycm9yKS4KICAgICAgICAiIiIKICAgICAgICBtZXNzYWdlcyA9IFsKICAgICAgICAgICAgU3lzdGVtTWVzc2FnZShjb250ZW50PXNlbGYuU1lTVEVNX1BST01QVCksCiAgICAgICAgICAgIEh1bWFuTWVzc2FnZShjb250ZW50PXN0YXRlLnF1ZXN0aW9uKSwKICAgICAgICBdCiAgICAgICAgcmVzcG9uc2U6IE9wdGlvbmFsW0FJTWVzc2FnZV0gPSBOb25lCgogICAgICAgIGZvciBfIGluIHJhbmdlKHNlbGYubWF4X2l0ZXJhdGlvbnMpOgogICAgICAgICAgICByZXNwb25zZSA9IHNlbGYuX2xsbV93aXRoX3Rvb2xzLmludm9rZShtZXNzYWdlcykKICAgICAgICAgICAgbWVzc2FnZXMuYXBwZW5kKHJlc3BvbnNlKQoKICAgICAgICAgICAgaWYgbm90IGdldGF0dHIocmVzcG9uc2UsICJ0b29sX2NhbGxzIiwgTm9uZSk6CiAgICAgICAgICAgICAgICBicmVhayAgIyBMTE0gcHJvZHVjZWQgYSBmaW5hbCBhbnN3ZXIKCiAgICAgICAgICAgIGZvciB0YyBpbiByZXNwb25zZS50b29sX2NhbGxzOgogICAgICAgICAgICAgICAgbmFtZSA9IHRjWyJuYW1lIl0KICAgICAgICAgICAgICAgIGFyZ3MgPSB0Y1siYXJncyJdCiAgICAgICAgICAgICAgICB0aWQgID0gdGNbImlkIl0KICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICByZXN1bHQgPSBzZWxmLl90b29sX21hcFtuYW1lXS5pbnZva2UoYXJncykgaWYgbmFtZSBpbiBzZWxmLl90b29sX21hcCBlbHNlIGYiVW5rbm93biB0b29sOiB7bmFtZX0iCiAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0ID0gZiJUb29sIGVycm9yOiB7ZX0iCiAgICAgICAgICAgICAgICBtZXNzYWdlcy5hcHBlbmQoVG9vbE1lc3NhZ2UoY29udGVudD1zdHIocmVzdWx0KSwgdG9vbF9jYWxsX2lkPXRpZCkpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcmVzcG9uc2UgPSBzZWxmLl9sbG1fd2l0aF90b29scy5pbnZva2UoCiAgICAgICAgICAgICAgICBtZXNzYWdlcyArIFtIdW1hbk1lc3NhZ2UoY29udGVudD0iUGxlYXNlIHByb3ZpZGUgeW91ciBmaW5hbCBhbnN3ZXIgbm93LiIpXQogICAgICAgICAgICApCgogICAgICAgIGFuc3dlciA9IHJlc3BvbnNlLmNvbnRlbnQgaWYgcmVzcG9uc2UgZWxzZSAiQ291bGQgbm90IGdlbmVyYXRlIGFuIGFuc3dlci4iCiAgICAgICAgcmV0dXJuIFJBR1N0YXRlKAogICAgICAgICAgICBxdWVzdGlvbj1zdGF0ZS5xdWVzdGlvbiwKICAgICAgICAgICAgcmV0cmlldmVkX2RvY3M9c3RhdGUucmV0cmlldmVkX2RvY3MsCiAgICAgICAgICAgIGFuc3dlcj1hbnN3ZXIsCiAgICAgICAgKQoKCiMg4pSA4pSAIEdyYXBoQnVpbGRlciDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKY2xhc3MgR3JhcGhCdWlsZGVyOgogICAgIiIiQXNzZW1ibGVzIGFuZCBjb21waWxlcyB0aGUgTGFuZ0dyYXBoIFJBRyB3b3JrZmxvdy4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgcmV0cmlldmVyLCBsbG0pOgogICAgICAgIHNlbGYubm9kZXMgPSBSQUdOb2RlcyhyZXRyaWV2ZXIsIGxsbSkKICAgICAgICBzZWxmLmdyYXBoID0gTm9uZQoKICAgIGRlZiBidWlsZChzZWxmKToKICAgICAgICBidWlsZGVyID0gU3RhdGVHcmFwaChSQUdTdGF0ZSkKICAgICAgICBidWlsZGVyLmFkZF9ub2RlKCJyZXRyaWV2ZXIiLCBzZWxmLm5vZGVzLnJldHJpZXZlX2RvY3MpCiAgICAgICAgYnVpbGRlci5hZGRfbm9kZSgicmVzcG9uZGVyIiwgc2VsZi5ub2Rlcy5nZW5lcmF0ZV9hbnN3ZXIpCiAgICAgICAgYnVpbGRlci5zZXRfZW50cnlfcG9pbnQoInJldHJpZXZlciIpCiAgICAgICAgYnVpbGRlci5hZGRfZWRnZSgicmV0cmlldmVyIiwgInJlc3BvbmRlciIpCiAgICAgICAgYnVpbGRlci5hZGRfZWRnZSgicmVzcG9uZGVyIiwgRU5EKQogICAgICAgIHNlbGYuZ3JhcGggPSBidWlsZGVyLmNvbXBpbGUoKQogICAgICAgIHByaW50KCLinIUgTGFuZ0dyYXBoIGNvbXBpbGVkIOKGkiByZXRyaWV2ZXIg4p6cIHJlc3BvbmRlciDinpwgRU5EIikKICAgICAgICByZXR1cm4gc2VsZi5ncmFwaAoKICAgIGRlZiBydW4oc2VsZiwgcXVlc3Rpb246IHN0cikgLT4gZGljdDoKICAgICAgICBpZiBzZWxmLmdyYXBoIGlzIE5vbmU6CiAgICAgICAgICAgIHNlbGYuYnVpbGQoKQogICAgICAgIHJldHVybiBzZWxmLmdyYXBoLmludm9rZShSQUdTdGF0ZShxdWVzdGlvbj1xdWVzdGlvbikpCg==",
    "utils/__init__.py": "",
    "utils/logger.py": "aW1wb3J0IGxvZ2dpbmcKaW1wb3J0IG9zCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lCgpMT0dTX0RJUiA9ICJsb2dzIgpvcy5tYWtlZGlycyhMT0dTX0RJUiwgZXhpc3Rfb2s9VHJ1ZSkKCkxPR19GSUxFID0gb3MucGF0aC5qb2luKExPR1NfRElSLCBmImxvZ197ZGF0ZXRpbWUubm93KCkuc3RyZnRpbWUoJyVZLSVtLSVkJyl9LmxvZyIpCgpsb2dnaW5nLmJhc2ljQ29uZmlnKAogICAgZmlsZW5hbWU9TE9HX0ZJTEUsCiAgICBmb3JtYXQ9IiUoYXNjdGltZSlzIC0gJShsZXZlbG5hbWUpcyAtICUobWVzc2FnZSlzIiwKICAgIGxldmVsPWxvZ2dpbmcuSU5GTywKKQoKCmRlZiBnZXRfbG9nZ2VyKG5hbWUpOgogICAgbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIobmFtZSkKICAgIGxvZ2dlci5zZXRMZXZlbChsb2dnaW5nLklORk8pCiAgICByZXR1cm4gbG9nZ2VyCg==",
    "utils/custom_exception.py": "aW1wb3J0IHN5cwoKCmNsYXNzIEN1c3RvbUV4Y2VwdGlvbihFeGNlcHRpb24pOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG1lc3NhZ2U6IHN0ciwgZXJyb3JfZGV0YWlsOiBFeGNlcHRpb24gPSBOb25lKToKICAgICAgICBzZWxmLmVycm9yX21lc3NhZ2UgPSBzZWxmLmdldF9kZXRhaWxlZF9lcnJvcl9tZXNzYWdlKG1lc3NhZ2UsIGVycm9yX2RldGFpbCkKICAgICAgICBzdXBlcigpLl9faW5pdF9fKHNlbGYuZXJyb3JfbWVzc2FnZSkKCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgZ2V0X2RldGFpbGVkX2Vycm9yX21lc3NhZ2UobWVzc2FnZSwgZXJyb3JfZGV0YWlsKToKICAgICAgICBfLCBfLCBleGNfdGIgPSBzeXMuZXhjX2luZm8oKQogICAgICAgIGZpbGVfbmFtZSAgID0gZXhjX3RiLnRiX2ZyYW1lLmZfY29kZS5jb19maWxlbmFtZSBpZiBleGNfdGIgZWxzZSAiVW5rbm93biBGaWxlIgogICAgICAgIGxpbmVfbnVtYmVyID0gZXhjX3RiLnRiX2xpbmVubyBpZiBleGNfdGIgZWxzZSAiVW5rbm93biBMaW5lIgogICAgICAgIHJldHVybiAoCiAgICAgICAgICAgIGYie21lc3NhZ2V9IHwgRXJyb3I6IHtlcnJvcl9kZXRhaWx9ICIKICAgICAgICAgICAgZiJ8IEZpbGU6IHtmaWxlX25hbWV9IHwgTGluZToge2xpbmVfbnVtYmVyfSIKICAgICAgICApCgogICAgZGVmIF9fc3RyX18oc2VsZik6CiAgICAgICAgcmV0dXJuIHNlbGYuZXJyb3JfbWVzc2FnZQo=",
    "streamlit_app.py": "IyBGbGlwa2FydCBBSSBBc3Npc3RhbnQg4oCUIFN0cmVhbWxpdCBVSQojIFJ1biB3aXRoOiBzdHJlYW1saXQgcnVuIHN0cmVhbWxpdF9hcHAucHkKIyBOT1RFOiBGb3IgQ29sYWIgdHVubmVsIHVzZSBweW5ncm9rIChzZWUgcmVmZXJlbmNlIG5vdGVib29rKS4KCmltcG9ydCBvcywgdGltZSwgdXVpZAppbXBvcnQgc3RyZWFtbGl0IGFzIHN0CmZyb20gbGFuZ2NoYWluX2dyb3EgaW1wb3J0IENoYXRHcm9xCmZyb20gbGFuZ2NoYWluX2NvbW11bml0eS52ZWN0b3JzdG9yZXMgaW1wb3J0IEZBSVNTCmZyb20gbGFuZ2NoYWluX2h1Z2dpbmdmYWNlIGltcG9ydCBIdWdnaW5nRmFjZUVtYmVkZGluZ3MKZnJvbSBsYW5nY2hhaW5fdGV4dF9zcGxpdHRlcnMgaW1wb3J0IFJlY3Vyc2l2ZUNoYXJhY3RlclRleHRTcGxpdHRlcgpmcm9tIGxhbmdjaGFpbl9jb3JlLm1lc3NhZ2VzIGltcG9ydCBIdW1hbk1lc3NhZ2UsIFN5c3RlbU1lc3NhZ2UKZnJvbSBmbGlwa2FydC5kYXRhX2NvbnZlcnRlciBpbXBvcnQgRGF0YUNvbnZlcnRlcgoKR1JPUV9BUElfS0VZICAgID0gb3MuZW52aXJvbi5nZXQoIkdST1FfQVBJX0tFWSIsICIiKQpMTE1fTU9ERUwgICAgICAgPSAibGxhbWEtMy4xLThiLWluc3RhbnQiCkVNQkVEX01PREVMICAgICA9ICJhbGwtTWluaUxNLUw2LXYyIgoKc3Quc2V0X3BhZ2VfY29uZmlnKHBhZ2VfdGl0bGU9IkZsaXBrYXJ0IEFJIENoYXRib3QiLCBwYWdlX2ljb249IvCfm5IiLCBsYXlvdXQ9ImNlbnRlcmVkIikKc3QudGl0bGUoIvCfm5IgRmxpcGthcnQgQUkgQXNzaXN0YW50IikKc3QuY2FwdGlvbigiUG93ZXJlZCBieSBHcm9xIMK3IEZBSVNTIMK3IEh1Z2dpbmdGYWNlIEVtYmVkZGluZ3MiKQoKQHN0LmNhY2hlX3Jlc291cmNlKHNob3dfc3Bpbm5lcj0i8J+UjSBCdWlsZGluZyBrbm93bGVkZ2UgYmFzZSAuLi4iKQpkZWYgaW5pdF9waXBlbGluZSgpOgogICAgZG9jcyAgID0gRGF0YUNvbnZlcnRlcigiZGF0YS9mbGlwa2FydF9wcm9kdWN0X3Jldmlldy5jc3YiKS5jb252ZXJ0KCkKICAgIGNodW5rcyA9IFJlY3Vyc2l2ZUNoYXJhY3RlclRleHRTcGxpdHRlcigKICAgICAgICBjaHVua19zaXplPTUwMCwgY2h1bmtfb3ZlcmxhcD01MAogICAgKS5zcGxpdF9kb2N1bWVudHMoZG9jcykKICAgIGVtYmVkICA9IEh1Z2dpbmdGYWNlRW1iZWRkaW5ncyhtb2RlbF9uYW1lPUVNQkVEX01PREVMKQogICAgdnMgICAgID0gRkFJU1MuZnJvbV9kb2N1bWVudHMoY2h1bmtzLCBlbWJlZCkKICAgIHJldCAgICA9IHZzLmFzX3JldHJpZXZlcihzZWFyY2hfa3dhcmdzPXsiayI6IDR9KQogICAgbGxtICAgID0gQ2hhdEdyb3EobW9kZWw9TExNX01PREVMLCBhcGlfa2V5PUdST1FfQVBJX0tFWSwgdGVtcGVyYXR1cmU9MCkKICAgIHJldHVybiBsbG0sIHJldCwgbGVuKGNodW5rcykKCmxsbSwgcmV0cmlldmVyLCBuX2NodW5rcyA9IGluaXRfcGlwZWxpbmUoKQpzdC5zdWNjZXNzKGYi4pyFIHtuX2NodW5rc30gcmV2aWV3IGNodW5rcyBpbmRleGVkIikKCmlmICJtZXNzYWdlcyIgbm90IGluIHN0LnNlc3Npb25fc3RhdGU6CiAgICBzdC5zZXNzaW9uX3N0YXRlLm1lc3NhZ2VzICAgICAgID0gW10KICAgIHN0LnNlc3Npb25fc3RhdGUucmVxdWVzdF9jb3VudCAgPSAwCiAgICBzdC5zZXNzaW9uX3N0YXRlLnByZWRpY3Rpb25fY291bnQgPSAwCgppZiBzdC5zaWRlYmFyLmJ1dHRvbigi8J+UhCBOZXcgQ2hhdCIpOgogICAgc3Quc2Vzc2lvbl9zdGF0ZS5tZXNzYWdlcyA9IFtdCiAgICBzdC5yZXJ1bigpCgpmb3IgbXNnIGluIHN0LnNlc3Npb25fc3RhdGUubWVzc2FnZXM6CiAgICB3aXRoIHN0LmNoYXRfbWVzc2FnZShtc2dbInJvbGUiXSk6CiAgICAgICAgc3QubWFya2Rvd24obXNnWyJjb250ZW50Il0pCgp1c2VyX2lucHV0ID0gc3QuY2hhdF9pbnB1dCgiQXNrIG1lIGFib3V0IEZsaXBrYXJ0IHByb2R1Y3RzIC4uLiIpCgppZiB1c2VyX2lucHV0OgogICAgc3Quc2Vzc2lvbl9zdGF0ZS5yZXF1ZXN0X2NvdW50ICs9IDEKICAgIHN0LnNlc3Npb25fc3RhdGUubWVzc2FnZXMuYXBwZW5kKHsicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiB1c2VyX2lucHV0fSkKICAgIHdpdGggc3QuY2hhdF9tZXNzYWdlKCJ1c2VyIik6CiAgICAgICAgc3QubWFya2Rvd24odXNlcl9pbnB1dCkKCiAgICB3aXRoIHN0LmNoYXRfbWVzc2FnZSgiYXNzaXN0YW50Iik6CiAgICAgICAgd2l0aCBzdC5zcGlubmVyKCLwn6SWIFRoaW5raW5nIC4uLiIpOgogICAgICAgICAgICBkb2NzICAgID0gcmV0cmlldmVyLmludm9rZSh1c2VyX2lucHV0KQogICAgICAgICAgICBjb250ZXh0ID0gIlxuXG4iLmpvaW4oCiAgICAgICAgICAgICAgICBmIlt7aX1dIFByb2R1Y3Q6IHtkLm1ldGFkYXRhLmdldCgncHJvZHVjdF9uYW1lJywnPycpfVxuUmV2aWV3OiB7ZC5wYWdlX2NvbnRlbnR9IgogICAgICAgICAgICAgICAgZm9yIGksIGQgaW4gZW51bWVyYXRlKGRvY3MsIDEpCiAgICAgICAgICAgICkgb3IgIk5vIHJlbGV2YW50IHJldmlld3MgZm91bmQuIgogICAgICAgICAgICByZXNwb25zZSA9IGxsbS5pbnZva2UoWwogICAgICAgICAgICAgICAgU3lzdGVtTWVzc2FnZShjb250ZW50PSgKICAgICAgICAgICAgICAgICAgICAiWW91IGFyZSBhIEZsaXBrYXJ0IGFzc2lzdGFudC4gQW5zd2VyIHVzaW5nIE9OTFkgdGhlIGNvbnRleHQgYmVsb3cuICIKICAgICAgICAgICAgICAgICAgICAiSWYgY29udGV4dCBpcyBpbnN1ZmZpY2llbnQsIHNheSBzbyBwb2xpdGVseS4iCiAgICAgICAgICAgICAgICApKSwKICAgICAgICAgICAgICAgIEh1bWFuTWVzc2FnZShjb250ZW50PWYiQ29udGV4dDpcbntjb250ZXh0fVxuXG5RdWVzdGlvbjoge3VzZXJfaW5wdXR9IiksCiAgICAgICAgICAgIF0pCiAgICAgICAgICAgIHJlcGx5ID0gcmVzcG9uc2UuY29udGVudAogICAgICAgIHN0Lm1hcmtkb3duKHJlcGx5KQogICAgc3Quc2Vzc2lvbl9zdGF0ZS5wcmVkaWN0aW9uX2NvdW50ICs9IDEKICAgIHN0LnNlc3Npb25fc3RhdGUubWVzc2FnZXMuYXBwZW5kKHsicm9sZSI6ICJhc3Npc3RhbnQiLCAiY29udGVudCI6IHJlcGx5fSkKCnN0LmRpdmlkZXIoKQpjb2wxLCBjb2wyID0gc3QuY29sdW1ucygyKQpjb2wxLm1ldHJpYygi8J+TpSBSZXF1ZXN0cyIsICAgIHN0LnNlc3Npb25fc3RhdGUucmVxdWVzdF9jb3VudCkKY29sMi5tZXRyaWMoIvCfpJYgUHJlZGljdGlvbnMiLCBzdC5zZXNzaW9uX3N0YXRlLnByZWRpY3Rpb25fY291bnQpCg=="
}

written = []
for path, b64content in _FILES_B64.items():
    content = base64.b64decode(b64content).decode("utf-8")
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    written.append(path)

print(f"✅ {len(written)} source files written to disk:")
for p in sorted(written):
    print(f"   {p:<52s}  {os.path.getsize(p):>5} bytes")


✅ 9 source files written to disk:
   flipkart/__init__.py                                      0 bytes
   flipkart/config.py                                      518 bytes
   flipkart/data_converter.py                              563 bytes
   flipkart/data_ingestion.py                             1634 bytes
   flipkart/rag_agent.py                                  5434 bytes
   streamlit_app.py                                       3196 bytes
   utils/__init__.py                                         0 bytes
   utils/custom_exception.py                               710 bytes
   utils/logger.py                                         436 bytes


---
## 🔑 Stage 1 — Configuration & API Keys
Only **2 keys** needed — both already saved in your Colab Secrets.


In [3]:
import os, sys
sys.path.insert(0, ".")

from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
HF_TOKEN     = userdata.get('HuggingFace_Token')

os.environ["GROQ_API_KEY"] = GROQ_API_KEY or ""
os.environ["HF_TOKEN"]     = HF_TOKEN or ""

with open(".env", "w") as f:
    f.write(f"GROQ_API_KEY={GROQ_API_KEY or ''}\n")
    f.write(f"HF_TOKEN={HF_TOKEN or ''}\n")

from flipkart.config import Config

print("✅  Stage 1 complete")
print(f"    Model  : {Config.LLM_MODEL}")
print(f"    GROQ   : {'SET ✓' if GROQ_API_KEY else 'NOT SET ✗'}")
print(f"    HF     : {'SET ✓' if HF_TOKEN else 'NOT SET ✗'}")


✅  Stage 1 complete
    Model  : llama-3.1-8b-instant
    GROQ   : SET ✓
    HF     : SET ✓


---
## 📊 Stage 2 — Data Converter
Reads `flipkart_product_review.csv` and converts each row into a LangChain `Document`  
with `review` as `page_content` and `product_title` as metadata.


In [5]:
# ── Choose ONE upload method ───────────────────────────────────────────────

# OPTION A: Upload from local machine
# ─────────────────────────────────────────────────────────────────────────
# from google.colab import files
# import shutil
# uploaded = files.upload()
# shutil.move(list(uploaded.keys())[0], "data/flipkart_product_review.csv")

# OPTION B: Mount Google Drive
# ─────────────────────────────────────────────────────────────────────────
# from google.colab import drive, import shutil
# drive.mount('/content/drive')
# shutil.copy('/content/drive/MyDrive/flipkart_product_review.csv',
#             'data/flipkart_product_review.csv')

# ── Verify ────────────────────────────────────────────────────────────────
import os, pandas as pd

DATA_PATH = "data/flipkart_product_review.csv"
if not os.path.exists(DATA_PATH):
    print("⚠️  File not found: data/flipkart_product_review.csv")
    print("   Uncomment one of the upload options above.")
else:
    df = pd.read_csv(DATA_PATH)
    print(f"✅  File found!")
    print(f"    Shape   : {df.shape}")
    print(f"    Columns : {list(df.columns)}")
    display(df[["product_title", "review"]].head(3))


✅  File found!
    Shape   : (450, 5)
    Columns : ['product_id', 'product_title', 'rating', 'summary', 'review']


,product_title,review
0,BoAt Rockerz 235v2 with ASAP charging Version ...,1-more flexible2-bass is very high3-sound clar...
1,BoAt Rockerz 235v2 with ASAP charging Version ...,Super sound and good looking I like that prize
2,BoAt Rockerz 235v2 with ASAP charging Version ...,Very much satisfied with the device at this pr...


In [6]:
from flipkart.data_converter import DataConverter

DATA_PATH = "data/flipkart_product_review.csv"

print("⚙️  Running DataConverter...")
converter = DataConverter(file_path=DATA_PATH)
docs      = converter.convert()

print(f"\n✅  Stage 2 complete")
print(f"    Total documents : {len(docs)}")
print(f"\n    Sample entries:")
for i, doc in enumerate(docs[:3], 1):
    print(f"    [{i}] Product : {doc.metadata.get('product_name','N/A')[:60]}")
    print(f"         Review  : {doc.page_content[:100]}...")

with open("outputs/stage2_data_converter.txt", "w", encoding="utf-8") as f:
    f.write(f"Total documents: {len(docs)}\n\n")
    for i, doc in enumerate(docs[:5], 1):
        f.write(f"[{i}] {doc.metadata}\n{doc.page_content[:200]}\n\n")

print("\n💾  Saved → outputs/stage2_data_converter.txt")


⚙️  Running DataConverter...

✅  Stage 2 complete
    Total documents : 450

    Sample entries:
    [1] Product : BoAt Rockerz 235v2 with ASAP charging Version 5.0 Bluetooth 
         Review  : 1-more flexible2-bass is very high3-sound clarity is good 4-battery back up to 6 to 8 hour's 5-main ...
    [2] Product : BoAt Rockerz 235v2 with ASAP charging Version 5.0 Bluetooth 
         Review  : Super sound and good looking I like that prize...
    [3] Product : BoAt Rockerz 235v2 with ASAP charging Version 5.0 Bluetooth 
         Review  : Very much satisfied with the device at this price point being from an awesome brand. Design wise , I...

💾  Saved → outputs/stage2_data_converter.txt


---
## 🗄️ Stage 3 — Data Ingestion (FAISS Vector Store)
Embeds all review documents with `BAAI/bge-base-en-v1.5` (runs **locally on Colab GPU/CPU**)  
and builds a FAISS index in memory.

> ⏳ First run downloads ~440 MB embedding model — takes ~2 min.  
> The index lives in RAM for the session — no external DB needed.


In [7]:
from flipkart.data_ingestion import DataIngestor

print("⚙️  DataIngestor — building FAISS index...")
ingestor  = DataIngestor()
retriever = ingestor.ingest(csv_path="data/flipkart_product_review.csv")

# Sanity-check: test similarity search
test_docs = retriever.invoke("best smartphone under 15000")
print(f"\n✅  Stage 3 complete")
print(f"    Retriever type  : {type(retriever).__name__}")
print(f"\n🔍  Test search — 'best smartphone under 15000':")
for i, d in enumerate(test_docs, 1):
    print(f"    [{i}] {d.metadata.get('product_name','?')[:60]}")
    print(f"         {d.page_content[:100]}...")

with open("outputs/stage3_vectorstore.txt", "w", encoding="utf-8") as f:
    f.write("FAISS index built (local, no external DB)\n\n")
    for i, d in enumerate(test_docs, 1):
        f.write(f"[{i}] {d.metadata}\n{d.page_content[:200]}\n\n")

print("\n💾  Saved → outputs/stage3_vectorstore.txt")


⚙️  DataIngestor — building FAISS index...
  🔄 Loading embedding model: BAAI/bge-base-en-v1.5 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  ✅ Embedding model loaded
  ✂️  450 rows → 450 chunks
  🔍 Building FAISS index over 450 chunks ...
  ✅ FAISS index ready

✅  Stage 3 complete
    Retriever type  : VectorStoreRetriever

🔍  Test search — 'best smartphone under 15000':
    [1] realme Buds Q Bluetooth Headset
         First of all, size of the product is very small and handy, it can easily fit in your pocket. Battery...
    [2] BoAt Airdopes 131 Bluetooth Headset
         Got the device in 2 days.Fast delivery by flipkart.Sound quality is the best at this price range.Eve...
    [3] OnePlus Bullets Wireless Z Bluetooth Headset
         Looking nice product...I got just 1day delivery..11th I ordered...12th I received...in this price ra...
    [4] OnePlus Bullets Wireless Z Bass Edition Bluetooth Headset
         Awesome looking and stylish all my friends are asking me for testing this product and some of them a...

💾  Saved → outputs/stage3_vectorstore.txt


---
## 🤖 Stage 4 — RAG Agent (LangGraph ReAct)
Builds a **LangGraph** pipeline:
```
retriever node → responder node (Groq-safe bind_tools + manual ToolMessage loop) → END
```
The LLM autonomously calls `search_flipkart_reviews` to look up product reviews  
before composing a final answer.


In [8]:
from flipkart.config import Config
from flipkart.rag_agent import GraphBuilder

print("🤖  Initialising LLM...")
llm = Config.get_llm()
print(f"    Model: {Config.LLM_MODEL} (Groq) ✓")

print("\n🔗  Building LangGraph...")
graph_builder = GraphBuilder(retriever=retriever, llm=llm)
graph_builder.build()

print(f"\n✅  Stage 4 complete")
print(f"    Tool     : search_flipkart_reviews ✓")
print(f"    Memory   : stateless per-call (RAGState) ✓")
print(f"    Loop     : bind_tools + manual ToolMessage (Groq-safe) ✓")


🤖  Initialising LLM...
    Model: llama-3.1-8b-instant (Groq) ✓

🔗  Building LangGraph...
✅ LangGraph compiled → retriever ➜ responder ➜ END

✅  Stage 4 complete
    Tool     : search_flipkart_reviews ✓
    Memory   : stateless per-call (RAGState) ✓
    Loop     : bind_tools + manual ToolMessage (Groq-safe) ✓


---
## 🧪 Stage 5 — Test Queries
Run live product queries through the full pipeline. Results saved to `outputs/`.


In [9]:
import time

QUERY_1 = "What do customers say about Samsung smartphones?"

print(f"📨  Query 1: {QUERY_1}")
print("=" * 65)
t0       = time.time()
result_1 = graph_builder.run(QUERY_1)
elapsed  = time.time() - t0

print(result_1["answer"])
print(f"\n⏱️  {elapsed:.2f}s  |  {len(result_1['retrieved_docs'])} docs retrieved")

with open("outputs/stage5_query1_samsung.txt", "w", encoding="utf-8") as f:
    f.write(f"Query: {QUERY_1}\n\nAnswer:\n{result_1['answer']}\n")
print("💾  Saved → outputs/stage5_query1_samsung.txt")


📨  Query 1: What do customers say about Samsung smartphones?
I don't have enough information on Samsung smartphones. The search results provided are for OnePlus Bullets Wireless Z Bass Edition Bluetooth Headset and U&I Titanic Series - Low Price Bluetooth Neckband Bluetooth Headset. Please contact customer care for more information on Samsung smartphones.

⏱️  0.62s  |  4 docs retrieved
💾  Saved → outputs/stage5_query1_samsung.txt


In [10]:
QUERY_2 = "Which earphones have the best battery life according to reviews?"

print(f"📨  Query 2: {QUERY_2}")
print("=" * 65)
t0       = time.time()
result_2 = graph_builder.run(QUERY_2)
elapsed  = time.time() - t0

print(result_2["answer"])
print(f"\n⏱️  {elapsed:.2f}s  |  {len(result_2['retrieved_docs'])} docs retrieved")

with open("outputs/stage5_query2_earphones.txt", "w", encoding="utf-8") as f:
    f.write(f"Query: {QUERY_2}\n\nAnswer:\n{result_2['answer']}\n")
print("💾  Saved → outputs/stage5_query2_earphones.txt")


📨  Query 2: Which earphones have the best battery life according to reviews?
Based on the reviews, it seems that the BoAt Airdopes 131 Bluetooth Headset and the OnePlus Bullets Wireless Z Bluetooth Headset have the best battery life, with one review mentioning 15 hours of playback. The BoAt Airdopes 131 also received praise for its sound quality and battery life. The OnePlus Bullets Wireless Z Bass Edition also received a positive review, but there is no specific mention of battery life in this review.

⏱️  0.68s  |  4 docs retrieved
💾  Saved → outputs/stage5_query2_earphones.txt


In [11]:
QUERY_3 = "Are Redmi phones good value for money based on customer feedback?"

print(f"📨  Query 3: {QUERY_3}")
print("=" * 65)
t0       = time.time()
result_3 = graph_builder.run(QUERY_3)
elapsed  = time.time() - t0

print(result_3["answer"])
print(f"\n⏱️  {elapsed:.2f}s  |  {len(result_3['retrieved_docs'])} docs retrieved")

with open("outputs/stage5_query3_redmi.txt", "w", encoding="utf-8") as f:
    f.write(f"Query: {QUERY_3}\n\nAnswer:\n{result_3['answer']}\n")
print("💾  Saved → outputs/stage5_query3_redmi.txt")


📨  Query 3: Are Redmi phones good value for money based on customer feedback?
Based on the search results, it seems that the reviews are not about Redmi phones but about other products such as Bluetooth headsets. Therefore, I don't have enough information on Redmi phones to provide a summary of customer feedback on their value for money. Please contact customer care for more information.

⏱️  0.90s  |  4 docs retrieved
💾  Saved → outputs/stage5_query3_redmi.txt


In [12]:
# ── Interactive: try your own query ──────────────────────────────────────
YOUR_QUERY = "Which laptop is best for college students under 50000?"  # ← edit this

print(f"📨  Your Query: {YOUR_QUERY}")
print("=" * 65)
t0            = time.time()
result_custom = graph_builder.run(YOUR_QUERY)
elapsed       = time.time() - t0

print(result_custom["answer"])
print(f"\n⏱️  {elapsed:.2f}s  |  {len(result_custom['retrieved_docs'])} docs retrieved")

with open("outputs/stage5_query_custom.txt", "w", encoding="utf-8") as f:
    f.write(f"Query: {YOUR_QUERY}\n\nAnswer:\n{result_custom['answer']}\n")
print("💾  Saved → outputs/stage5_query_custom.txt")


📨  Your Query: Which laptop is best for college students under 50000?
Based on the search results, it seems that there are several good options for laptops under 50,000. However, I couldn't find any specific recommendations for laptops under 50,000 that are suitable for college students.

If you're looking for a laptop for college, I would recommend considering the following factors:

* Processor: Look for a laptop with a recent-generation processor from Intel or AMD.
* RAM: Ensure the laptop has at least 8GB of RAM for smooth performance.
* Storage: Consider a laptop with a solid-state drive (SSD) for faster loading times and better overall performance.
* Display: A Full HD display is a good starting point, but consider a laptop with a higher resolution if you want better image quality.
* Battery Life: Look for a laptop with a battery life of at least 8 hours to ensure you can use it throughout the day.

Some popular laptop brands that offer good options under 50,000 include:

* Dell


In [13]:
# ── View retrieved docs for the last query ───────────────────────────────
print(f"📄  Retrieved docs for: '{YOUR_QUERY}'")
print("=" * 65)
for i, doc in enumerate(result_custom["retrieved_docs"], 1):
    product = doc.metadata.get("product_name", "Unknown")
    print(f"\n[{i}] Product: {product}")
    print(f"    {doc.page_content[:250]}...")


📄  Retrieved docs for: 'Which laptop is best for college students under 50000?'

[1] Product: realme Buds Q Bluetooth Headset
    First of all, size of the product is very small and handy, it can easily fit in your pocket. Battery backup is good as of now.Call quality is average, not expecting more...Sound quality is good for bass lovers, but not crispy. I'm using sony Headset ...

[2] Product: BoAt BassHeads 100 Wired Headset
    One of the best under 500 if you can get ,started loving from the first day.Clarity of microphone is awesome.Bass is really good with equalisation of treble .Build quality is beyond level.Thanks Flipkart ❤...

[3] Product: OnePlus Bullets Wireless Z Bluetooth Headset
    Thanks to flipkart fot super fast delivery in our rural area.Got it within 3 days.Super earphones using it for online pg classes,comes with handy backup hrs.5/5 for sound quality4/5 for bass(not as high,for competitor like sony310,or realme wireless)...

[4] Product: realme Buds Wireless Blue

---
## 💾 Stage 6 — Save All Files & Export ZIP


In [14]:
import zipfile, os, glob
from datetime import datetime

ZIP_NAME = f"Flipkart_RAG_Project_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"

INCLUDE = [
    "flipkart/*.py",
    "utils/*.py",
    "outputs/*.txt",
    "streamlit_app.py",
    "requirements.txt",
    ".env",
]

print(f"📦  Creating {ZIP_NAME} ...")
added = []
with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as zf:
    for pattern in INCLUDE:
        for fp in sorted(glob.glob(pattern)):
            if os.path.isfile(fp):
                zf.write(fp)
                added.append(fp)
                print(f"    + {fp}")

size_kb = os.path.getsize(ZIP_NAME) / 1024
print(f"\n✅  {ZIP_NAME}  ({size_kb:.1f} KB)  |  {len(added)} files")

try:
    from google.colab import files
    files.download(ZIP_NAME)
    print("📥  Download triggered.")
except ImportError:
    print(f"📦  File ready at: {os.path.abspath(ZIP_NAME)}")


📦  Creating Flipkart_RAG_Project_20260406_035536.zip ...
    + flipkart/__init__.py
    + flipkart/config.py
    + flipkart/data_converter.py
    + flipkart/data_ingestion.py
    + flipkart/rag_agent.py
    + utils/__init__.py
    + utils/custom_exception.py
    + utils/logger.py
    + outputs/stage2_data_converter.txt
    + outputs/stage3_vectorstore.txt
    + outputs/stage5_query1_samsung.txt
    + outputs/stage5_query2_earphones.txt
    + outputs/stage5_query3_redmi.txt
    + outputs/stage5_query_custom.txt
    + streamlit_app.py
    + requirements.txt
    + .env

✅  Flipkart_RAG_Project_20260406_035536.zip  (9.5 KB)  |  17 files


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥  Download triggered.


---
## 📋 Stage 7 — Summary & Log Viewer


In [15]:
import glob, os

DIV = "=" * 65
print(DIV)
print("  Flipkart RAG Recommender — Run Summary")
print(DIV)

print("\n📁  Source .py files:")
for f in sorted(glob.glob("**/*.py", recursive=True)):
    if "__pycache__" not in f:
        print(f"   {f:<52s}  {os.path.getsize(f):>6} B")

print("\n📊  Outputs:")
for f in sorted(glob.glob("outputs/*")):
    print(f"   {f:<52s}  {os.path.getsize(f):>6} B")

print("\n📜  Latest log entries:")
logs = sorted(glob.glob("logs/*.log"), reverse=True)
if logs:
    with open(logs[0], encoding="utf-8") as lf:
        lines = lf.read().strip().splitlines()
    for line in lines[-15:]:
        print(f"   {line}")
else:
    print("   (no logs yet)")

print(f"\n{DIV}")
print("  ✅  All stages complete — Flipkart RAG Agent operational!")
print(DIV)


  Flipkart RAG Recommender — Run Summary

📁  Source .py files:
   flipkart/__init__.py                                       0 B
   flipkart/config.py                                       518 B
   flipkart/data_converter.py                               563 B
   flipkart/data_ingestion.py                              1634 B
   flipkart/rag_agent.py                                   5434 B
   streamlit_app.py                                        3196 B
   utils/__init__.py                                          0 B
   utils/custom_exception.py                                710 B
   utils/logger.py                                          436 B

📊  Outputs:
   outputs/stage2_data_converter.txt                       1240 B
   outputs/stage3_vectorstore.txt                          1097 B
   outputs/stage5_query1_samsung.txt                        362 B
   outputs/stage5_query2_earphones.txt                      512 B
   outputs/stage5_query3_redmi.txt                          395 B
